In [24]:
import pandas as pd
import numpy as np
import boto3
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker import Session
import io
import sagemaker.amazon.common as smac
import os
from sagemaker.amazon.amazon_estimator import get_image_uri

In [25]:
df = pd.read_csv("loan_dataset.csv")
df.head()

,Gender,Marital_Status,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount_INR,Loan_Term_Years,Credit_History,Existing_Loans,Loan_Status
0,1,1,1,0,60154,57252,2241607,5,1,0,1
1,0,0,0,0,140466,74736,303525,10,1,2,1
2,1,0,0,1,43253,29851,1506744,10,0,0,0
3,1,0,0,1,95940,8159,2435311,30,1,0,1
4,1,0,0,1,185476,57475,920616,10,1,2,1


In [26]:
df.shape

(150000, 11)

In [27]:
X = df.drop(columns=["Loan_Status"])
y = df[["Loan_Status"]]

In [28]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [29]:
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [35]:
X_train = X_train.astype("float32")
y_train = y_train.astype("float32")

X_test  = X_test.astype("float32")
y_test  = y_test.astype("float32")

In [36]:
X_train

array([[ 1.,  0.,  1., ..., 10.,  1.,  0.],
       [ 1.,  1.,  0., ..., 30.,  1.,  2.],
       [ 0.,  1.,  0., ..., 25.,  0.,  0.],
       ...,
       [ 1.,  0.,  1., ..., 25.,  0.,  0.],
       [ 0.,  1.,  1., ..., 20.,  1.,  0.],
       [ 1.,  0.,  1., ..., 25.,  1.,  0.]], dtype=float32)

In [37]:
X_test

array([[ 0.,  0.,  0., ..., 15.,  1.,  1.],
       [ 1.,  0.,  1., ...,  5.,  1.,  2.],
       [ 0.,  0.,  1., ...,  5.,  1.,  0.],
       ...,
       [ 0.,  1.,  1., ..., 15.,  0.,  2.],
       [ 1.,  1.,  1., ..., 30.,  1.,  1.],
       [ 1.,  0.,  0., ..., 15.,  1.,  0.]], dtype=float32)

In [39]:
y_train = y_train[:,0]

In [40]:
y_train

array([1., 1., 1., ..., 1., 1., 1.], dtype=float32)

In [42]:
y_test = y_test[:,0]

In [43]:
sagemaker_session = sagemaker.Session()

bucket_name = "loan-predict"

prefix="linear-predict"

role= sagemaker.get_execution_role()

print(role)

arn:aws:iam::118903272754:role/service-role/AmazonSageMaker-ExecutionRole-20260301T152362


In [44]:
X_train = np.array(X_train)

In [45]:
bucket_name

'loan-predict'

In [46]:
buf = io.BytesIO()

smac.write_numpy_to_dense_tensor(buf, X_train, y_train)
buf.seek(0)

0

In [47]:
key = "loan-data"

boto3.resource("s3").Bucket(bucket_name).Object(os.path.join(prefix, "train", key)).upload_fileobj(buf)

s3_train_data = f"s3://{bucket_name}/{prefix}/train/{key}"

print("Data Uploaded:", s3_train_data)

Data Uploaded: s3://loan-predict/linear-predict/train/loan-data


In [48]:
X_test = np.array(X_test)

buf = io.BytesIO()

smac.write_numpy_to_dense_tensor(buf,X_test,y_test)
buf.seek(0)

key = "loan-date-test"

boto3.resource("s3").Bucket(bucket_name).Object(os.path.join(prefix, "test", key)).upload_fileobj(buf)

s3_train_data = f"s3://{bucket_name}/{prefix}/test/{key}"

print("Data Uploaded:", s3_train_data)

Data Uploaded: s3://loan-predict/linear-predict/test/loan-date-test


In [49]:
output_loaction = f"s3://{bucket_name}/{prefix}/output"
output_loaction

's3://loan-predict/linear-predict/output'

In [50]:
container = sagemaker.image_uris.retrieve("linear-learner",boto3.Session().region_name)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [51]:
linear = sagemaker.estimator.Estimator(container,role,instance_count=1,instance_type="ml.m5.large",output_path = output_loaction,sagemaker_session=sagemaker_session)

In [52]:
linear.set_hyperparameters(feature_dim=10,predictor_type="regressor",mini_batch_size=4,epochs=6,num_models=32,loss="absolute_loss")

In [53]:
linear.fit({"train":s3_train_data})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-01-11-38-55-151


2026-03-01 11:38:58 Starting - Starting the training job...
2026-03-01 11:39:12 Starting - Preparing the instances for training...
2026-03-01 11:39:33 Downloading - Downloading input data...
2026-03-01 11:40:18 Downloading - Downloading the training image.........
2026-03-01 11:41:44 Training - Training image download completed. Training in progress.Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/01/2026 11:41:49 INFO 140057837754176] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss':

In [55]:
linear_regresor = linear.deploy(initial_instance_count=1,instance_type="ml.m5.large")

INFO:sagemaker:Creating model with name: linear-learner-2026-03-01-11-53-11-730
INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-01-11-53-11-730
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-01-11-53-11-730


-------!

In [57]:
linear_regresor.serializer = sagemaker.serializers.CSVSerializer()
linear_regresor.deserializer = sagemaker.deserializers.JSONDeserializer()

In [60]:
result = linear_regresor.predict(X_test)

In [61]:
result

{'predictions': [{'score': 1.000004529953003},
  {'score': 0.9999977946281433},
  {'score': 0.9999905228614807},
  {'score': 0.9999881982803345},
  {'score': -2.4237935576820746e-05},
  {'score': -2.034776844084263e-05},
  {'score': 0.9999919533729553},
  {'score': 0.9999794960021973},
  {'score': 0.9999815225601196},
  {'score': 0.9999939799308777},
  {'score': 0.9999855160713196},
  {'score': 0.9999948143959045},
  {'score': 0.9999849200248718},
  {'score': 0.9999756217002869},
  {'score': 0.9999884366989136},
  {'score': 0.9999917149543762},
  {'score': 0.9999935030937195},
  {'score': 0.9999693632125854},
  {'score': 0.9999883770942688},
  {'score': 0.9999827146530151},
  {'score': 0.9999880194664001},
  {'score': -1.2662278095376678e-05},
  {'score': 0.9999834299087524},
  {'score': 0.9999733567237854},
  {'score': 0.9999752640724182},
  {'score': -2.5526705940137617e-05},
  {'score': -1.5947418432915583e-05},
  {'score': -1.087788950826507e-05},
  {'score': -2.5192452085320838e-0

In [70]:
import boto3

runtime = boto3.client("sagemaker-runtime")

payload = "1.0,1.0,1.0,0.0,60154.0,57252.0,2241607.0,5.0,1.0,0.0\n"

response = runtime.invoke_endpoint(
    EndpointName="linear-learner-2026-03-01-11-53-11-730",
    ContentType="text/csv",
    Body=payload
)

print(response["Body"].read().decode())

{"predictions": [{"score": 0.999977707862854}]}


In [66]:
import boto3

runtime = boto3.client("sagemaker-runtime")

payload = "1.0,0.0,0.0,1.0,43253.0,29851.0,1506744.0,10.0,0.0,0.0\n"

response = runtime.invoke_endpoint(
    EndpointName="linear-learner-2026-03-01-11-53-11-730",
    ContentType="text/csv",
    Body=payload
)

print(response["Body"].read().decode())

{"predictions": [{"score": -1.5765619536978193e-05}]}


In [72]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime", region_name="eu-north-1")

payload = "1.0,0.0,0.0,1.0,43253.0,29851.0,1506744.0,10.0,0.0,0.0\n"

response = runtime.invoke_endpoint(
    EndpointName="linear-learner-2026-03-01-11-53-11-730",
    ContentType="text/csv",
    Body=payload
)

result = response["Body"].read().decode()
print("Raw output:", result)

# Parse prediction
prediction = json.loads(result)["predictions"][0]["score"]

if prediction > 0:
    print("Approved")
else:
    print("Rejected")

Raw output: {"predictions": [{"score": -1.5765619536978193e-05}]}
Rejected


In [73]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime", region_name="eu-north-1")

payload = "1.0,1.0,1.0,0.0,60154.0,57252.0,2241607.0,5.0,1.0,0.0\n"

response = runtime.invoke_endpoint(
    EndpointName="linear-learner-2026-03-01-11-53-11-730",
    ContentType="text/csv",
    Body=payload
)

result = response["Body"].read().decode()
print("Raw output:", result)

# Parse prediction
prediction = json.loads(result)["predictions"][0]["score"]

if prediction > 0:
    print("Approved")
else:
    print("Rejected")

Raw output: {"predictions": [{"score": 0.999977707862854}]}
Approved
